## Global variables and basic load functions

In [6]:
import scipy.io
import io
 
from data_loader import DataLoader

import numpy as np
from utils import *
from matplotlib import pyplot as plt
import numpy.random as rng
import pandas as pd
from scipy.stats import spearmanr

import os

In [7]:
home_dir = os.path.abspath(os.path.curdir)
data_dir = os.path.join(home_dir, '..', 'Data')
ultralight = os.path.join(data_dir, 'GG-Dataset-Tulio-Ultralight')

In [8]:
def load_data(rats):
    return DataLoader(ultralight, 'Hpc', rats=[rats])

In [47]:
# # global variables
rats = ['Rat08', 'Rat09', 'Rat10', 'Rat11']
# rat_id = 3
# rat = rats[rat_id]

# # load data
# data = load_data(rat)
# spks = data.get_spk()
# pos = data._get_pos()

# Session variables, ripples, spike matrix

In [10]:
def session_vars(session_idx):
    sessions_spks = list(spks[rat].keys()) 
    sessions_data = list(data.events[rat].keys())
   
    sessions = np.intersect1d(sessions_spks, sessions_data)
    sessions.sort()
    session = sessions[session_idx]

    session_neurons = spks[rat][session]['all']
    neuron_ids = list(session_neurons.keys())
    n_neurons = len(neuron_ids)

    idx_start = list(data.events[rat][session]['cat']['data']).index('beginning run')
    start_time = data.events[rat][session]['cat']['ts'][idx_start]

    idx_end = list(data.events[rat][session]['cat']['data']).index('end run')
    end_time = data.events[rat][session]['cat']['ts'][idx_end]

    airpuff_times = data.events[rat][session]['puf']['ts']
    airpuff_times = [t for t in airpuff_times if t > start_time and t < end_time]

    return session, session_neurons, n_neurons, neuron_ids, start_time, end_time, airpuff_times

In [11]:
def position_vars(session):
    
    pos = data._get_pos()
    x = pos[rat][session]['x']
    y = pos[rat][session]['y']
    time = pos[rat][session]['Time']

    return x, y, time

In [12]:
def spk_matrix_init(session_neurons, start_time, end_time):
    length = end_time - start_time
    n_neurons = len(session_neurons)
    spike_matrix = np.zeros((n_neurons, length), dtype=int)
    neuron_ids = session_neurons.keys()
    
    for i, neuron_id in enumerate(neuron_ids):
        spike_times = np.floor(session_neurons[neuron_id]).astype(int)
        spike_times_windowed = spike_times[(spike_times >= start_time) & (spike_times < end_time)]- start_time #<- comment out to not make it start at 0
        spike_matrix[i, spike_times_windowed] = 1 #set corresponding columns to 1 where spikes occurred
    return spike_matrix   

In [13]:
def get_ripples(session):
    mat = scipy.io.loadmat(os.path.join(ultralight, rat, session, 'SWSripples.mat'))
    column_names = ['rip_start', 'rip_stop']
    ripples = pd.DataFrame(mat['swsripples']*1000, columns = column_names)

    return ripples

In [14]:
def spk_puff_matrix(airpuff_events, time_frame, session_neurons):
    n_neurons = len(session_neurons.keys())
    n_puffs = len(airpuff_events)

    spk_puff_matrix = np.zeros((n_puffs, n_neurons, time_frame))
    
    for t, start_t in enumerate(airpuff_events):
        end_t = start_t + time_frame
        for i, (neuron_id, neuron_spk_time) in enumerate(session_neurons.items()):
            idx = np.where((neuron_spk_time >= start_t) & (neuron_spk_time < end_t))[0]
            if len(idx)==0:
                continue
                
            spks_during_timeframe = neuron_spk_time[idx]
            spks_during_timeframe = np.floor((spks_during_timeframe - start_t)).astype(int)

            spk_puff_matrix[t, i, spks_during_timeframe] = 1

    return spk_puff_matrix

In [15]:
def spk_ripple_matrix(ripple_events, time_frame, session_neurons):
    n_neurons = len(session_neurons.keys())
    n_ripples = len(ripple_events)

    spk_rip_matrix = np.zeros((n_ripples, n_neurons, time_frame))
    
    for t, (start_t, end_t) in enumerate(zip(ripple_events['rip_start'], ripple_events['rip_stop'])):
        for i, (neuron_id, neuron_spk_time) in enumerate(session_neurons.items()):
            idx = np.where((neuron_spk_time >= start_t) & (neuron_spk_time < end_t))
            if len(idx)==0:
                continue
                
            spks_during_rip = neuron_spk_time[idx]
            spks_during_rip = np.floor((spks_during_rip - start_t)).astype(int)

            spk_rip_matrix[t, i, spks_during_rip] = 1

    return spk_rip_matrix

In [16]:
def spk_matrix_to_ranks_without_trigger(spk_matrix):
    
    avg_spk_time = []
    
    for neuron_id in range(spk_matrix.shape[1]):
        spike_times = np.where(spk_matrix[:,neuron_id] == 1)[0]
        if len(spike_times) > 0:  #check if neuron has spikes
            avg_spk_time.append(np.mean(spike_times))  
        else:
            avg_spk_time.append(np.nan) 

    avg_spk_time = np.array(avg_spk_time) 
    neuron_sorted_idx = np.argsort(avg_spk_time)
    ranks = np.full([spk_matrix.shape[1], spk_matrix.shape[1]], 0)  # Shape: [neurons x neurons] (ranks are in columns)

    for rank, neuron in enumerate(neuron_sorted_idx):
        if not np.isnan(avg_spk_time[neuron]):
            ranks[neuron, rank] = 1

    # Return the ranks matrix (with neuron indices as row labels, and rank positions as column labels)
    return ranks

In [17]:
def spk_matrix_to_ranks(spk_matrix, trigger_neuron_idx): #output is the neuron idx in the place related to the trigger neuron
    # wat als de trigger neuron niet actief is -> dan nog steeds opschuiven, en hoe zit dat met de MI? 
    # spk_matrix = [trials, neurons, timewindow (150)]
    n_trials, n_neurons = spk_matrix.shape[0], spk_matrix.shape[1]
    
    ranks = np.full((n_trials, n_neurons*2+1), np.nan)  # Shape: [trials x neurons*2+1]
    for trial in range(n_trials):
        avg_spk_time = []
        for neuron_idx in range(n_neurons):
            spike_times = np.where(spk_matrix[trial, neuron_idx] == 1)[0]  
            if len(spike_times) > 0:  
                avg_spk_time.append(np.mean(spike_times))  
                # avg_spk_time.append(np.min(spike_times))  
            else:
                avg_spk_time.append(np.nan)  

        avg_spk_time = np.array(avg_spk_time) 
        neuron_sorted = np.argsort(avg_spk_time)
        
        valid_neurons = [neuron_idx for neuron_idx in neuron_sorted if not np.isnan(avg_spk_time[neuron_idx])]

        if trigger_neuron_idx in valid_neurons:
            base_rank = valid_neurons.index(trigger_neuron_idx) # base rank gets index of trigger neuron in valid neurons
            ranks[trial, n_neurons] = trigger_neuron_idx  # Trigger neuron rank is 0
        else:
            base_rank = None
              

        for relative_rank, neuron_idx in enumerate(valid_neurons): #relative rank is index in valid neurons 
            if base_rank is not None:  
                idx = relative_rank - base_rank + n_neurons
                ranks[trial, idx] = neuron_idx
            else:  
                ranks[trial, :] = np.nan # hier dus nog goed naar kijken!!!
    return ranks



# Analysis functions

In [18]:
def aligned_firing_ranks2occ_matrix(firing_ranks):
    """
    Constructs an occurrence matrix from firing ranks where occ[x, y] represents how often neuron x had rank y.
    Intended for relative ranks use case.
    Input:  - firing_ranks: Array containing for each rank in each trial which neuron had this rank in this trial
    Output: - occ:          Occurrence matrix
    """
    n_neurons = int((firing_ranks.shape[1]+1)/2) # krijgt eigenlijk 1 neuron teveel
    occ = np.zeros((n_neurons, n_neurons*2-2), dtype=int) #[n_neurons, n_neurons * 2 - 2) -> voor en na trigger, zonder de trigger
    # Loop over trials
    for trial in firing_ranks:
        # Loop over ranks:
        for rank, neuron in enumerate(trial): # rank is index, neuron is wat er staat (= neuron ID)
            if np.isnan(neuron) or rank == n_neurons-1: # je telt inactieve neurons en de trigger neuron niet
                continue
            # For not nan rank, neuron pairs, increment the respective square in the occurrence matrix
            if rank > n_neurons-1:
                occ[int(neuron), rank-1] += 1
            else:
                occ[int(neuron), rank] += 1
    return occ

In [19]:
def firing_ranks2occ_matrix(firing_ranks):
    """
    Constructs an occurrence matrix from firing ranks where occ[x, y] represents how often neuron x had rank y.
    Input:  - firing_ranks: Array containing for each rank in each trial which neuron had this rank in this trial
    Output: - occ:          Occurrence matrix
    """
    n_neurons = firing_ranks.shape[1]
    occ = np.zeros((n_neurons, n_neurons), dtype=int)
    # Loop over trials
    for trial in firing_ranks:
        # Loop over ranks:
        for rank, neuron in enumerate(trial):
            if np.isnan(neuron):
                continue
            # For not nan rank, neuron pairs, increment the respective square in the occurrence matrix|
            occ[int(neuron), rank] += 1
    return occ

In [20]:
def MI(M):
    """
    Function made by Tom Has.
    Calculates the mutual information between the 2 axes of a matrix.
    Input:  - M:  2d matrix
    Output: - MI: Mutual information
    """
    sizex = M.shape[0]
    sizey = M.shape[1]
    total = np.sum(M)
    p_Y = np.sum(M, axis=0) / total
    p_X = np.sum(M, axis=1) / total
    # MI = sum over x and y: p(x,y) * log(p(x,y) / (p(x) * p(y)))
    #    = sum over x and y: M(x,y)/total * log((M(x,y) * total) / (p_X(x) * p_Y(y)))
    MI = sum([sum([(
        0 if M[x, y] == 0 else
        M[x, y] / total * np.log((M[x, y] / total) / (p_X[x] * p_Y[y]))
    ) for x in range(sizex)]) for y in range(sizey)])
    return MI

In [21]:
def shuffle_ranks(ranks, trigger_neuron_idx):
    shuffled_ranks = ranks.copy()
    n_trials, n_neurons = ranks.shape[0], (ranks.shape[1]-1)//2

    for trial in range(n_trials):
        active_neuron_idxs = [idx for idx, val in enumerate(ranks[trial]) if not np.isnan(val) and idx != n_neurons]
        active_neurons = [ranks[trial, idx] for idx in active_neuron_idxs]
    
        rng.shuffle(active_neurons)
    
        for idx, neuron in zip(active_neuron_idxs, active_neurons):
            shuffled_ranks[trial, idx] = neuron

        #shuffled_ranks[trial, trigger_neuron_idx] = ranks[trial, trigger_neuron_idx]
    
    return shuffled_ranks

In [22]:
def MI_trigger_neuron(spk_matrix, trigger_neuron_idx, n_iterations):
    ranks = spk_matrix_to_ranks(spk_matrix, trigger_neuron_idx)
    
    occ_actual = aligned_firing_ranks2occ_matrix(ranks)
    MI_actual = MI(occ_actual)
    
    surrogate_mi = np.empty(n_iterations, dtype=float)
    for i in range(n_iterations):
        shuffled_ranks = shuffle_ranks(ranks, trigger_neuron_idx)
        # shuffled_ranks = shuffle_ranks_trigger_removed(ranks, trigger_neuron_idx)
        occ_surrogate = aligned_firing_ranks2occ_matrix(shuffled_ranks)
        surrogate_mi[i] = MI(occ_surrogate)
    
    significance_boundary = np.percentile(surrogate_mi, 95)
    
    return MI_actual, occ_actual, surrogate_mi, significance_boundary


In [23]:
def surrogate_MI_neurons(spk_matrix, n_iterations):
    n_neurons = spk_matrix.shape[1]
    MI = np.empty(n_neurons)
    MI_surrogate = np.empty((n_neurons, n_iterations))
    boundaries = np.empty(n_neurons)
    occ_per_trigger = np.zeros((n_neurons, n_neurons+1, n_neurons*2))
    for n in range(n_neurons): 
        print(f"Starting computation of the MI, OCC... of neuron {n}")
        MI[n], occ_per_trigger[n], MI_surrogate[n], boundaries[n] = MI_trigger_neuron(spk_matrix, n, n_iterations)
        
    return MI, occ_per_trigger, MI_surrogate, boundaries

In [24]:
def get_significant_neurons(mi_values, boundaries):
    significant_neurons = np.full((len(mi_values)), np.nan)
    for idx in range(len(mi_values)):
        mi_neuron = mi_values[idx]
        boundary = boundaries[idx]
        if mi_neuron > boundary and mi_neuron != 0:
            significant_neurons[idx] = mi_neuron

    return significant_neurons

In [25]:
def get_sequence_from_occ(original_mi, original_boundary, occ_list):
    significant_neurons = get_significant_neurons(original_mi, original_boundary)
    removed_neurons = []
    list_of_remaining_occ = occ_list.copy()
    while significant_neurons.size > 0:
        if np.all(np.isnan(significant_neurons) | (significant_neurons == 0)):
            print("No more significant neurons.")
            break

        neuron_to_delete_idx = np.nanargmax(significant_neurons)
        
        removed_neurons.append(neuron_to_delete_idx)

        new_MI = np.empty(len(original_mi))
        for i, occ_value in enumerate(list_of_remaining_occ):
            list_of_remaining_occ[i][neuron_to_delete_idx] = 0
            list_of_remaining_occ[neuron_to_delete_idx] = 0 
            new_MI[i] = MI(occ_value)

        significant_neurons = get_significant_neurons(new_MI, original_boundary)
       
    return removed_neurons                
            

In [26]:
def sort_occ_matrix(occ_matrix, mean_rank=False):   
    """"
    Modifies an occ matrix to only active neurons and 'used' rank positions. 
    Also, it sorts the matrix, either based on mean rank, or on mode rank.
    """
    # Find active neurons
    active_neurons = np.where(np.any(occ_matrix>0, axis=1))[0]

    # Find non-zero rank range
    active_ranks = np.where(np.any(occ_matrix!=0, axis=0))[0]
    if len(active_ranks)<=1:
        rank_limit_1 = 1
        rank_limit_2 = 2
    else:
        rank_limit_1 = active_ranks[1]
        rank_limit_2 = active_ranks[-1]

    # Pick 'active' rows of occ_matrix, for relevant rank range
    occ_matrix = occ_matrix[active_neurons, rank_limit_1:rank_limit_2+1]
    
    # Compute max rank
    max_ranks = np.argmax(occ_matrix, axis=1)
    
    # Compute mean rank
    mean_ranks = np.zeros(len(active_neurons))
    for ineuron, neuron in enumerate(active_neurons):
        mean_ranks[ineuron] = np.mean(occ_matrix[ineuron,:]*np.arange(rank_limit_1, rank_limit_2+1))
    
    
    # Return sorted occ matrix
    if mean_rank:
        return [occ_matrix[np.argsort(mean_ranks),:], np.argsort(mean_ranks), active_neurons]
    else:
        return [occ_matrix[np.argsort(max_ranks),:], np.argsort(max_ranks), active_neurons]

# Plot functions

In [27]:
def spike_diagram(neuron_ids, spk_matrix_slice):
    plt.figure(figsize=(8, 6))
    n_trials, n_neurons, n_time_steps = spk_matrix_slice.shape

    for i, neuron_id in enumerate(neuron_ids):
        # Extract the spike data for the current neuron across all trials
        neuron_spikes = spk_matrix_slice[:, i, :]  # Shape: (10, 150)

        # Find spike times for each trial
        for trial_idx in range(n_trials):
            spike_times = np.where(neuron_spikes[trial_idx, :] >= 1)[0]  # Spikes in trial `trial_idx`
            plt.plot(spike_times, i + np.ones(len(spike_times)), 'k|', markersize=10)

    # Set up axis labels and title
    plt.yticks(ticks=np.arange(len(neuron_ids)), labels=neuron_ids)
    plt.xlabel('Time in ms')
    plt.ylabel('Neuron ID')
    plt.title(f'Spike Diagram with {n_trials} Trials')
    plt.show()

In [28]:
def plot_running_track(x, start_t, end_t, total_time, safe_time):

    airpuffs = (data.events[rat][s]['puf']['ts'] - start_t) / 1000 / 60
    safe_times = (safe_time - start_t) / 1000 / 60
    rrw = (data.events[rat][s]['rrw']['ts'] - start_t) / 1000 / 60
    lrw = (data.events[rat][s]['lrw']['ts'] - start_t) / 1000 / 60
    
    time_in_min = (total_time - start_t) / 1000 / 60
    
    airpuff_x_positions = np.interp(airpuffs, time_in_min, x)
    airpuff_safe_positions = np.interp(safe_times, time_in_min, x)
    rrw_x_positions = np.interp(rrw, time_in_min, x)
    lrw_x_positions = np.interp(lrw, time_in_min, x)
    
    plt.figure(figsize=(6, 8))
    plt.plot(x, (time-start_t)/1000/60, label='X-position', linewidth=0.5, color='royalblue')
    plt.plot(airpuff_x_positions, airpuffs, 'o', color='red', label='Airpuff', markersize=2)
    # plt.plot(rrw_x_positions, rrw, 'o', color='yellow', label='RRW', markersize=2)
    # plt.plot(lrw_x_positions, lrw, 'o', color='green', label='LRW', markersize=2)
    plt.plot(airpuff_safe_positions, safe_times, 'o', color='green', label='Safe location', markersize=2)
    
    plt.title('X position vs Time (during the run)')
    plt.xlabel('X position')
    plt.ylabel('Time in m')
    plt.ylim((0, (end_t-start_t)/1000/60))
    # plt.legend()


In [29]:
def find_safe_moments(s, start_t, end_t):
    
    x, y, time = position_vars(s)
    
    rrw_ts = (data.events[rat][s]['rrw']['ts'])
    lrw_ts = (data.events[rat][s]['lrw']['ts'])
    rrw = [t for t in rrw_ts if t > start_t and t < end_t]
    lrw = [t for t in lrw_ts if t > start_t and t < end_t]
    all_rw = np.sort(rrw + lrw)
    
    time_run = [index for index, t in enumerate(time) if t > start_t and t < end_t]
    x_run = x[time_run[0]:time_run[-1]]
    y_run = y[time_run[0]:time_run[-1]]
    
    airpuffs = data.events[rat][s]['puf']['ts']
    puffs_checked = []
    
    if isinstance(list(airpuffs)[0], str):
        for t in np.array(airpuffs):
            puffs_checked.append(float(t[:14]))
    else:
        puffs_checked = airpuffs
    
    puffs_checked = [t for t in puffs_checked if t > start_t and t < end_t]
    puff_locations = [x_run[np.absolute(time - i).argmin()] for i in puffs_checked]
    x_mean = np.mean(puff_locations)
    
    safe_ts = []
    
    for reward_i, reward_t in enumerate(all_rw[:-1]):  
        puff_list = [t for t in airpuffs if t > reward_t and t < all_rw[reward_i + 1]]
        
        if not np.any(puff_list):
            time_idxs = [idx for idx, t in enumerate(time) if t > reward_t and t < all_rw[reward_i + 1]]
            
            if np.any(time_idxs):
                x_trial = list(x[time_idxs[0]:time_idxs[-1]])
                time_trial = list(time)[time_idxs[0]:time_idxs[-1]]           
                
                if len(np.array(x_trial)) > 0:  
                    safe_t = time_trial[np.absolute(np.array(x_trial) - x_mean).argmin()]
                    safe_ts.append(safe_t)
    
    return safe_ts


In [30]:
def find_sleep_times(s):
    # to be able to compare ripples in pre/post sleep...

    idx_start_presleep = list(data.events[rat][s]['cat']['data']).index('beginning presleep')
    ts_start_presleep = data.events[rat][s]['cat']['ts'][idx_start_presleep]
    
    idx_end_presleep = list(data.events[rat][s]['cat']['data']).index('end presleep')
    ts_end_presleep = data.events[rat][s]['cat']['ts'][idx_end_presleep]
    
    idx_start_postsleep = list(data.events[rat][s]['cat']['data']).index('beginning postsleep')
    ts_start_postsleep = data.events[rat][s]['cat']['ts'][idx_start_postsleep]
    
    idx_end_postsleep = list(data.events[rat][s]['cat']['data']).index('end postsleep')
    ts_end_postsleep = data.events[rat][s]['cat']['ts'][idx_end_postsleep]

    return ts_start_presleep, ts_end_presleep, ts_start_postsleep, ts_end_postsleep

In [31]:
def get_significant_ranks_seq(occ, seq):

    indices_significant_neurons = seq
    n_significant_neurons = len(indices_significant_neurons)

    middle_index = occ.shape[2] // 2 + 1
    
    rank_matrix = np.full((n_significant_neurons, n_significant_neurons), np.nan)
    
    for i, viewed_neuron in enumerate(indices_significant_neurons):
        for j, other_neuron in enumerate(indices_significant_neurons):
            if viewed_neuron == other_neuron:
                continue
            
            occ_row = occ[viewed_neuron][other_neuron]

            if np.any(occ_row):
                viewed_occ_indices = np.nonzero(occ_row)[0]
                rank_counts = {}
                
                for idx in viewed_occ_indices:
                    rank = idx - middle_index
                    rank_counts[rank] = occ_row[idx]
                
                highest_occurrence_rank = max(rank_counts, key=rank_counts.get)
                rank_matrix[i, j] = highest_occurrence_rank
    
    return rank_matrix, indices_significant_neurons

In [32]:
def get_significant_ranks(occ, significant_neurons):

    indices_significant_neurons = np.where(~np.isnan(significant_neurons))[0] 
    n_significant_neurons = len(indices_significant_neurons)

    middle_index = occ.shape[2] // 2 + 1
    
    rank_matrix = np.full((n_significant_neurons, n_significant_neurons), np.nan)
    
    for i, viewed_neuron in enumerate(indices_significant_neurons):
        for j, other_neuron in enumerate(indices_significant_neurons):
            if viewed_neuron == other_neuron:
                continue
            
            occ_row = occ[viewed_neuron][other_neuron]

            if np.any(occ_row):
                viewed_occ_indices = np.nonzero(occ_row)[0]
                rank_counts = {}
                
                for idx in viewed_occ_indices:
                    rank = idx - middle_index
                    rank_counts[rank] = occ_row[idx]
                    # rank_counts[rank] = rank_counts.get(rank, 0) + 1
                
                highest_occurrence_rank = max(rank_counts, key=rank_counts.get)
                rank_matrix[i, j] = highest_occurrence_rank
    
    return rank_matrix, indices_significant_neurons

In [33]:
def get_weighted_values(session_neurons, airpuffs, pattern_template, time_stamp, window_size=100, step_size=20):
    results = {"time_points": [], 
               "values": []} 

    for i, puff in enumerate(airpuffs):
        start_time = int(puff)
        end_time = start_time + time_stamp
        empty_spk_matrix_count = 0

        for t in range(start_time, end_time - window_size + 1, step_size):
            spk_matrix = spk_matrix_init(session_neurons, t, t + window_size)

            spiking_neurons = np.where(spk_matrix.sum(axis=1) > 0)[0]

            if len(spiking_neurons) == 0:
                empty_spk_matrix_count += 1
                continue

            trigger = spiking_neurons[0]
            ranks = spk_matrix_to_ranks_projection(spk_matrix, trigger)

            if ranks.shape != pattern_template.shape:
                raise ValueError("Ranks and occ_template must have the same shape.")

            # match_score = np.nansum(ranks * pattern_template)
            template_norm = pattern_template.flatten() / np.linalg.norm(pattern_template.flatten())
            ranks_norm = ranks.flatten() / np.linalg.norm(np.nan_to_num(ranks.flatten()))
            
            cosine_similarity = np.dot(np.nan_to_num(template_norm), np.nan_to_num(ranks_norm))

            results["time_points"].append(t)  
            results["values"].append(cosine_similarity) 
            # results["values"].append(match_score) 

    return results


In [34]:
def create_template(occ, significant_neurons, seq_neurons):
    neuron_idx = np.nanargmax(significant_neurons)
    if neuron_idx not in seq_neurons:
        raise ValueError(f"Neuron {neuron_idx} is not in the sequence order.")
    
    occ_template = occ[neuron_idx].copy()
    for neuron_idx in range(occ_template.shape[0]):
        if neuron_idx not in seq_neurons:
            occ_template[neuron_idx, :] = 0  # Set all values in the row to 0

    add_column = np.zeros((len(significant_neurons), 1))

    occ_template = occ_template[:-1]
    occ_template = np.hstack((occ_template, add_column))
    
    return occ_template

In [35]:
def create_template_mirror(occ, significant_neurons, seq_neurons):
    neuron_idx = np.nanargmax(significant_neurons)
    if neuron_idx not in seq_neurons:
        raise ValueError(f"Neuron {neuron_idx} is not in the sequence order.")
    
    # Create the base template
    occ_template = occ[neuron_idx].copy()
    for neuron_idx in range(occ_template.shape[0]):
        if neuron_idx not in seq_neurons:
            occ_template[neuron_idx, :] = 0  # Set all values in the row to 0

    # Add a zero column at the end
    add_column = np.zeros((len(significant_neurons), 1))
    occ_template = np.hstack((occ_template, add_column))
    
    # Mirror the matrix across the middle column
    middle_column = occ_template.shape[1] // 2
    left_part = occ_template[:, :middle_column]
    right_part = np.flip(occ_template[:, middle_column:], axis=1)
    mirrored_occ_template = np.hstack((left_part, right_part))
    
    return mirrored_occ_template

In [36]:
def mean_ranks_to_seq(mean_ranks, seq_neuron_ids):
    sorted_indices = np.argsort(mean_ranks)  

    sorted_ranks = mean_ranks[sorted_indices]
    seq_neuron_ids = np.array(seq_neuron_ids)
    sorted_neuron_ids = seq_neuron_ids[sorted_indices]
    
    return sorted_neuron_ids, sorted_ranks

In [37]:
def spk_matrix_to_ranks_projection(spk_matrix, trigger_neuron_idx): #output is the neuron idx in the place related to the trigger neuron
    n_neurons = spk_matrix.shape[0]
    ranks = np.full((n_neurons, n_neurons*2+1), np.nan)  # Shape: [neurons x neurons*2+1]
    
    avg_spk_time = []
    for neuron_idx in range(n_neurons):
        spike_times = np.where(spk_matrix[neuron_idx] == 1)[0]
        if len(spike_times) > 0:  
            avg_spk_time.append(np.mean(spike_times))  
        else:
            avg_spk_time.append(np.nan)  
            
    avg_spk_time = np.array(avg_spk_time) 
    neuron_sorted = np.argsort(avg_spk_time)
    valid_neurons = [neuron_idx for neuron_idx in neuron_sorted if not np.isnan(avg_spk_time[neuron_idx])]

    if trigger_neuron_idx in valid_neurons:
        base_rank = valid_neurons.index(trigger_neuron_idx) # base rank gets index of trigger neuron in valid neurons
        ranks[trigger_neuron_idx, base_rank+n_neurons] = 1  # Trigger neuron rank is 0
    else:
        base_rank = None
          

    for relative_rank, neuron_idx in enumerate(valid_neurons): #relative rank is index in valid neurons 
        if base_rank is not None:  
            idx = relative_rank - base_rank + n_neurons
            ranks[neuron_idx, idx] = 1
        else:  
            idx = relative_rank + n_neurons
            ranks[neuron_idx, idx] = np.nan # hier dus nog goed naar kijken!!!
    return ranks

# Initialize variables

In [40]:
def initialize_test_train(session_idx, puff_not_safe, n_surrogates, test_window, train_window=150):
    
    session, session_neurons, n_neurons, neuron_ids, start_t, end_t, puffs = session_vars(session_idx)
    x, y, time = position_vars(session)
    safe_moments = find_safe_moments(session, start_t, end_t)

    # ripple variables
    ripple_events = get_ripples(session)
    start_presleep, end_presleep, start_postsleep, end_postsleep = find_sleep_times(session)
    pre_ripples = ripple_events.loc[(ripple_events['rip_start'] >= start_presleep) & (ripple_events['rip_stop'] <= end_presleep)]
    post_ripples = ripple_events.loc[(ripple_events['rip_start'] >= start_postsleep) & (ripple_events['rip_stop'] <= end_postsleep)]

    if puff_not_safe: 
        train_size = int(len(puffs) * 0.7)
        train_data = rng.choice(puffs, train_size, replace=False)
        test_data = np.setdiff1d(puffs, train_data)
        spk_airpuff_matrix = spk_puff_matrix(train_data, train_window, session_neurons)
    else:
        train_size = int(len(safe_moments) * 0.7)
        train_data = rng.choice(safe_moments, train_size, replace=False)
        test_data = np.setdiff1d(safe_moments, train_data)
        spk_airpuff_matrix = spk_puff_matrix(train_data, train_window, session_neurons)

    spk_rip_matrix = spk_ripple_matrix(ripple_events, train_window, session_neurons)

    # MI, OCC, surrogates & boundary for every neuron as trigger neuron
    print("Starting computation mi...")
    mi, occ, surrogates, boundary = surrogate_MI_neurons(spk_airpuff_matrix, n_surrogates)
    significant_neurons = get_significant_neurons(mi, boundary)
    if np.isnan(significant_neurons).all():
        print(f"No significant neurons in session {session}. Skipping analysis for this session.")
        return None, None, None, None, None, None, train_data, test_data
        
    ranks_distribution, seq_neurons = get_significant_ranks(occ, significant_neurons)
    print(f"sequence neurons are: {seq_neurons}")
    mean_ranks = np.nanmean(ranks_distribution, axis=1)
    seq_neurons, seq_order = mean_ranks_to_seq(mean_ranks, seq_neurons)
    print(f"sequence neurons are: {seq_neurons}")
    template_occ = create_template(occ, significant_neurons, seq_neurons)

    if puff_not_safe: 
        puff_results = get_weighted_values(session_neurons, test_data, template_occ, test_window)
        safe_results = get_weighted_values(session_neurons, safe_moments, template_occ, test_window)

    else: 
        puff_results = get_weighted_values(session_neurons, puffs, template_occ, test_window)
        safe_results = get_weighted_values(session_neurons, test_data, template_occ, test_window)
    
    pre_results = get_weighted_values(session_neurons, pre_ripples['rip_start'], template_occ, test_window)
    post_results = get_weighted_values(session_neurons, post_ripples['rip_start'], template_occ, test_window)
    
    return seq_neurons, seq_order, puff_results, safe_results, pre_results, post_results, train_data, test_data
    

In [41]:
def sequence_occ_and_project_train_test(session_idx, train_data, test_data, puff_not_safe, n_surrogates, test_window, train_window=150):
    session, session_neurons, n_neurons, neuron_ids, start_t, end_t, puffs = session_vars(session_idx)
    x, y, time = position_vars(session)
    safe_moments = find_safe_moments(session, start_t, end_t)
    safe_moments_before = [x - train_window for x in safe_moments]

    # ripple variables
    ripple_events = get_ripples(session)
    start_presleep, end_presleep, start_postsleep, end_postsleep = find_sleep_times(session)
    pre_ripples = ripple_events.loc[(ripple_events['rip_start'] >= start_presleep) & (ripple_events['rip_stop'] <= end_presleep)]
    post_ripples = ripple_events.loc[(ripple_events['rip_start'] >= start_postsleep) & (ripple_events['rip_stop'] <= end_postsleep)]
   
    spk_airpuff_matrix = spk_puff_matrix(train_data, train_window, session_neurons)
    spk_rip_matrix = spk_ripple_matrix(ripple_events, train_window, session_neurons)

    # MI, OCC, surrogates & boundary for every neuron as trigger neuron
    print("Starting computation mi...")
    mi, occ, surrogates, boundary = surrogate_MI_neurons(spk_airpuff_matrix, n_surrogates)
    significant_neurons = get_significant_neurons(mi, boundary)
    if np.isnan(significant_neurons).all():
        print(f"No significant neurons in session {session}. Skipping analysis for this session.")
        return None, None, None, None, None, None

    seq = get_sequence_from_occ(mi, boundary, occ)
    if len(seq) == 0:
        return None, None, None, None, None, None
    else: 
        ranks_distribution, seq_neurons = get_significant_ranks_seq(occ, seq)
        mean_ranks = np.nanmean(ranks_distribution, axis=1)
        seq_neurons, seq_order = mean_ranks_to_seq(mean_ranks, seq_neurons)
        print(seq_neurons)
        template_occ = create_template(occ, significant_neurons, seq_neurons)
        
    if puff_not_safe: 
        puff_results = get_weighted_values(session_neurons, test_data, template_occ, test_window)
        safe_results = get_weighted_values(session_neurons, safe_moments, template_occ, test_window)

    else: 
        puff_results = get_weighted_values(session_neurons, puffs, template_occ, test_window)
        safe_results = get_weighted_values(session_neurons, test_data, template_occ, test_window)

    #projected in everything: 
    puff_results_ = get_weighted_values(session_neurons, puffs, template_occ, test_window)
    safe_results_ = get_weighted_values(session_neurons, safe_moments, template_occ, test_window)
    
    pre_results = get_weighted_values(session_neurons, pre_ripples['rip_start'], template_occ, test_window)
    post_results = get_weighted_values(session_neurons, post_ripples['rip_start'], template_occ, test_window)
    
    return seq_neurons, seq_order, puff_results, safe_results, pre_results, post_results

In [49]:
def initialize_test_train_given(session_idx, train_data, test_data, puff_not_safe, n_surrogates, test_window, train_window=150):
    session, session_neurons, n_neurons, neuron_ids, start_t, end_t, puffs = session_vars(session_idx)
    x, y, time = position_vars(session)
    safe_moments = find_safe_moments(session, start_t, end_t)
    
    # ripple variables
    ripple_events = get_ripples(session)
    start_presleep, end_presleep, start_postsleep, end_postsleep = find_sleep_times(session)
    pre_ripples = ripple_events.loc[(ripple_events['rip_start'] >= start_presleep) & (ripple_events['rip_stop'] <= end_presleep)]
    post_ripples = ripple_events.loc[(ripple_events['rip_start'] >= start_postsleep) & (ripple_events['rip_stop'] <= end_postsleep)]
   
    spk_airpuff_matrix = spk_puff_matrix(train_data, train_window, session_neurons)
    spk_rip_matrix = spk_ripple_matrix(ripple_events, train_window, session_neurons)

    # MI, OCC, surrogates & boundary for every neuron as trigger neuron
    print("Starting computation mi...")
    mi, occ, surrogates, boundary = surrogate_MI_neurons(spk_airpuff_matrix, n_surrogates)
    significant_neurons = get_significant_neurons(mi, boundary)
    if np.isnan(significant_neurons).all():
        print(f"No significant neurons in session {session}. Skipping analysis for this session.")
        return None, None, None, None, None, None

    ranks_distribution, seq_neurons = get_significant_ranks(occ, significant_neurons)
    print(f"sequence neurons are: {seq_neurons}")
    mean_ranks = np.nanmean(ranks_distribution, axis=1)
    seq_neurons, seq_order = mean_ranks_to_seq(mean_ranks, seq_neurons)
    print(f"sequence neurons are: {seq_neurons}")
    # template_occ = create_template(occ, significant_neurons, seq_neurons)
    template_occ = create_template_mirror(occ, significant_neurons, seq_neurons)

    if puff_not_safe: 
        puff_results = get_weighted_values(session_neurons, test_data, template_occ, test_window)
        safe_results = get_weighted_values(session_neurons, safe_moments, template_occ, test_window)

    else: 
        puff_results = get_weighted_values(session_neurons, puffs, template_occ, test_window)
        safe_results = get_weighted_values(session_neurons, test_data, template_occ, test_window)
    
    pre_results = get_weighted_values(session_neurons, pre_ripples['rip_start'], template_occ, test_window)
    post_results = get_weighted_values(session_neurons, post_ripples['rip_start'], template_occ, test_window)
    
    return seq_neurons, seq_order, puff_results, safe_results, pre_results, post_results
    

In [43]:
import pickle

with open("results_rat_11_puff_cos_70-30_56101213.pkl", "rb") as pkl_file:
    rat11_puff = pickle.load(pkl_file)

with open("train_data_rat11_puff_56101213.pkl", "rb") as pkl_file:
    rat11_train = pickle.load(pkl_file)

with open("test_data_rat11_puff_56101213.pkl", "rb") as pkl_file:
    rat11_test = pickle.load(pkl_file)
